# Stage 1: Paper Search (abstracts)

Embeds each paper's title + abstract from `data/downloaded/metadata.csv` into the ChromaDB collection `papers` (saved in `data/chroma/`), then searches it with a natural-language question.

Run `Retrieval.ipynb` first to download papers.

In [ ]:
#%pip install -q chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [2]:
import csv
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# Input metadata (from Retrieval.ipynb) and where the vector database is saved (relative to notebooks/)
META_PATH = "../data/downloaded/metadata.csv"
CHROMA_DIR = "../data/chroma"

# Embedding model: turns text into 384 numbers that represent its meaning, fast and lightweight
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Vector database on disk; "papers" = one entry per paper, compared by cosine similarity
db = chromadb.PersistentClient(path=CHROMA_DIR)
papers = db.get_or_create_collection(
    "papers", embedding_function=embed_fn, configuration={"hnsw": {"space": "cosine"}}
)


def index_papers():
    """Add every paper in metadata.csv that isn't in the collection yet.
    Returns the number of papers added (0 if everything was already indexed)."""
    with open(META_PATH, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    existing = set(papers.get(ids=[r["id"] for r in rows])["ids"]) if rows else set()
    new_rows = [r for r in rows if r["id"] not in existing]
    if new_rows:
        papers.add(
            ids=[r["id"] for r in new_rows],
            documents=[f"{r['title']}. {r['abstract']}" for r in new_rows],  # the text that gets embedded
            metadatas=[{"title": r["title"], "url": r["url"], "published": r["published"]} for r in new_rows],
        )
    return len(new_rows)


def search(question, k=5):
    """Return the k papers whose title + abstract best match the question.
    score = cosine similarity (higher is more relevant, max 1.0)."""
    res = papers.query(query_texts=[question], n_results=min(k, papers.count()))
    return [
        {"id": pid, "title": meta["title"], "url": meta["url"], "score": round(1 - dist, 3)}
        for pid, meta, dist in zip(res["ids"][0], res["metadatas"][0], res["distances"][0])
    ]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# Index any new papers (safe to re-run; only new ones are added)
print(index_papers(), "new paper(s) indexed,", papers.count(), "total")

10 new paper(s) indexed, 10 total


In [4]:
# Ask a question
for r in search("How can we reduce the cost of running LLM agents?"):
    print(r["score"], r["id"], r["title"], r["url"])

0.426 2609.35760v1 TokenCast: Forecasting Token Consumption During LLM Agent Execution http://arxiv.org/abs/2609.35760v1
0.363 2609.35750v1 KV-streams for Efficient Compaction in Agentic Reinforcement Learning http://arxiv.org/abs/2609.35750v1
0.341 2609.35741v1 Shockingly Simple Self-retrospection Improves Agentic Models Without RL http://arxiv.org/abs/2609.35741v1
0.32 2609.35769v1 Telescopic Language Models http://arxiv.org/abs/2609.35769v1
0.29 2609.35751v1 How to Loop MoE: Flatten the Experts, Untie the Attention http://arxiv.org/abs/2609.35751v1
